In [1]:
import glob
from joblib import Parallel, delayed
#from catalog_plotter import plot_var, get_array
import numpy as np
import cartopy as cp
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import plotter_config as pc
import sim_name_remap as map
import netCDF4 as nc4
import os
import datetime as dt
from scipy.interpolate import interp1d
import pickle as pkl
import matplotlib.tri as tri
from plotter import plot_map
import jdcal
import utils
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

In [2]:

# Retreive cloud observations
cld_file = '/glade/work/wsmith/cloud_data_file_final.pkl'
with open(cld_file, 'rb') as f: cloud_jd, cloudlon, cloudlat, cloudalt, cloudtheta, trop_alt, trop_theta = pkl.load(f)
cloudprs = utils.alt2prs(cloudalt, 7.6, 1013.)


In [3]:

PLOTDIR = '/glade/u/home/wsmith/ASM_Plotter/plots/'
sim = '2022_ERA5_ZMKE_30m'
var = 'ZMMU'
date0 = '2022-08-18'
date1 = '2022-08-20'

# Set up the colorbar
if var == 'PCONVT': 
    levels = [100,150,200,250,300,350]   # For PCONVT
    cmap = plt.get_cmap('Reds_r')
if var == 'ZMMU': 
    levels = [0,2,5,10,20,40,60,80,100,120]  # For ZMMU
    cmap = plt.get_cmap('Reds')
if var == 'CO':
    levels = [60,80,100,120,140]  # For CO
    cmap = plt.get_cmap('Reds')

#cmap = ListedColormap(np.asarray([[250/256,150/256,0],[1,1,0],[0.6,0.6,0.6],[0.8,0.8,0.8]]))
#cmap.set_over([1,1,1])
#cmap.set_under([1,0,0])

norm = BoundaryNorm(levels, ncolors = cmap.N, clip = False)


In [5]:

FILES = sorted(glob.glob(map.remap[sim]))
FILES0 = [item for item in FILES if date0 in item][0]
find0 = list(FILES).index(FILES0)
FILES1 = [item for item in FILES if date1 in item][0]
find1 = list(FILES).index(FILES1)
FILES = FILES[find0:find1+1]

for FILE in FILES:

    print(FILE)
    
    file = nc4.Dataset(FILE)
    
    model_lon = file.variables['lon'][:]
    model_lat = file.variables['lat'][:]
    model_lev = file.variables['lev'][:]
    
    PCONVT = file.variables['PCONVT'][:]/100.
    ZMMU = file.variables['ZMMU'][:]*3600.  # Convert to kg/m2/hr
    CO = file.variables['CO'][:]*1e9   # Convert to ppbv
    date = file.variables['date'][:]
    datesec = file.variables['datesec'][:]
    
    model_jd = []
    for t in range(len(date)):
    
        curr_jd = sum(jdcal.gcal2jd(str(date[t])[0:4], str(date[t])[4:6], str(date[t])[6:8])) + datesec[t]/86400.
    
        cloud_tind = np.argmin(np.abs(cloud_jd - curr_jd))
    
        fig = plt.figure(figsize=(12,7))
        ax = fig.add_subplot(111, projection = ccrs.PlateCarree())
    
        if var == 'PCONVT': cf = ax.contourf(model_lon, model_lat, np.squeeze(PCONVT[t,:,:]), levels=levels, norm=norm, cmap=cmap, extend='both')
        if var == 'ZMMU': cf = ax.contourf(model_lon, model_lat, np.squeeze(ZMMU[t,14,:,:]), levels=levels, norm=norm, cmap=cmap, extend='both')     # ZMMU at ~200 hPa   
        if var == 'CO': cf = ax.contourf(model_lon, model_lat, np.squeeze(CO[t,14,:,:]), levels=levels, norm=norm, cmap=cmap, extend='both')     #  CO at ~200 hPa

        cld = ax.contour(cloudlon, cloudlat, cloudprs[cloud_tind,:,:], levels=[200], colors=['blue'], linewidths=3)
        
        ax.coastlines()
        
        cbax = fig.add_axes([0.17,0.14,0.7,0.02])
        cbar = plt.colorbar(cf, cax=cbax, orientation='horizontal')
        cbar.set_label(pc.var_info[var]['lab'] + ' (' + pc.var_info[var]['unit'] +  ')')
    
        ax.set_xlim([100,140])
        ax.set_ylim([25,45])
        
        ax.set_title(pc.var_info[var]['lab'] + ' and Observed Deep Convection at ' + str(date[t]) + '_' + str(round(datesec[t]/3600.)).zfill(2) + 'Z from ' + sim)

        plt.savefig(PLOTDIR + var + '_' + sim + '_' + str(date[t]) + '_' + str(int(datesec[t]/3600.)).zfill(2) + 'Z_' + str(int((datesec[t]/60.) % 60)).zfill(2) + 'min.png')                                               
        plt.close('all')
        

/glade/derecho/scratch/wsmith/archive/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic_withbb_RichConv.002.2022Aug_E5_ZMKE/atm/hist/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic_withbb_RichConv.002.2022Aug_E5_ZMKE.cam.h1.2022-08-18-01800.nc
/glade/derecho/scratch/wsmith/archive/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic_withbb_RichConv.002.2022Aug_E5_ZMKE/atm/hist/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic_withbb_RichConv.002.2022Aug_E5_ZMKE.cam.h1.2022-08-18-12600.nc
/glade/derecho/scratch/wsmith/archive/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic_withbb_RichConv.002.2022Aug_E5_ZMKE/atm/hist/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic_withbb_RichConv.002.2022Aug_E5_ZMKE.cam.h1.2022-08-18-23400.nc
/glade/derecho/scratch/wsmith/archive/FCnudged_f09.carmats16_cesm2.2.carma_trop_strat.2010_2021_cams5.3_so2_meic